# HIS flood engine and surrogate on Kaggle

Nothing needs uploading: the terrain, the storms and the interventions are generated from a seed, so this notebook clones the repository and rebuilds everything.

**Before you run anything**, open the panel on the right and set:

1. **Accelerator -> GPU T4 x2** (two GPUs halve generation; see `kaggle/HOWTO.md`)
2. **Internet -> On** — without this the clone in cell 1 fails with `Could not resolve host: github.com`

Then run cell 1, set the stage in cell 2, and run it. Full instructions, timings and troubleshooting: `kaggle/HOWTO.md` in the repo.

In [ ]:
# Cell 1 - fetch the code (safe to re-run; it fast-forwards an existing clone)
import os, subprocess, sys

REPO = "https://github.com/Elciiid/HIS.git"   # change if you forked it
WORK = "/kaggle/working"
HIS = f"{WORK}/HIS"

token = None
try:                                           # only needed if the repository is private
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    print("using the GITHUB_TOKEN secret")
except Exception as e:
    print(f"no GITHUB_TOKEN secret ({type(e).__name__}); assuming a public repository")

url = REPO if not token else REPO.replace("https://", f"https://x-access-token:{token}@")
if os.path.isdir(f"{HIS}/.git"):
    subprocess.run(["git", "-C", HIS, "fetch", "--depth", "50", "origin"], check=True)
    subprocess.run(["git", "-C", HIS, "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "50", url, HIS], check=True)
print(subprocess.run(["git", "-C", HIS, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)

## Cell 2 — run one stage

Change `STAGE` and run. One stage per session is the safe pattern; `k0` chains the four short ones.

| stage | what it does | how long |
|---|---|---|
| `k0` | environment, benchmarks + tests vs the local numbers, hardware timings, storage check | ~1–1.5 h |
| `workers` | measures storms/hour, cores busy and host RAM at 1, 2, 4 workers, and checks a parallel run is bit-identical to a serial one | ~1 h |
| `generate` | paired dataset, as much as `--budget-hours` allows (resumable) | fills the session |
| `train` | paired two-stage training, resumes from its checkpoint | fills the session |
| `evaluate` | Gate 1 / Report 2 evaluation of the trained surrogate | ~0.5 h |
| `diagnose` | A1 resolution arms, the volume correction, two-stage speed | ~1 h |

Run `workers` once before `generate` and pass what it recommends as `--workers`. `generate` needs a passing benchmark record for this configuration in the same session; it runs the analytical suite itself if there is none, which adds about half an hour.

In [ ]:
# Cell 2 - run a stage
STAGE = "k0"            # k0 | workers | generate | train | evaluate | diagnose
WORKERS = 2             # what the `workers` stage measured on T4 x2; ignored by short stages
BUDGET_HOURS = 10.5     # generation stops cleanly before this much wall time

!python {HIS}/kaggle/run.py --stage {STAGE} --workers {WORKERS} --budget-hours {BUDGET_HOURS} --config kaggle/configs/kaggle.json

## Cell 3 — what came out

Everything lands in `/kaggle/working/artifacts`. Reports are Markdown, measurements are JSON, and the dataset and model checkpoints are the large directories. When the notebook is committed (**Save & Run All**), all of it becomes the notebook's output, which the next session can attach as a dataset input.

In [ ]:
# Cell 3 - list what was produced, and show the headline reports
import subprocess
print(subprocess.run(["du", "-sh", "/kaggle/working/artifacts"], capture_output=True, text=True).stdout)
print(subprocess.run(["find", "/kaggle/working/artifacts", "-maxdepth", "2", "-name", "*.md"],
                     capture_output=True, text=True).stdout)
for name in ("verify_comparison.md", "kaggle_hardware.md", "storage_check.md",
             "worker_scaling.md", "validation_report.md"):
    p = f"/kaggle/working/artifacts/{name}"
    try:
        print("\n" + "=" * 80 + f"\n{name}\n" + "=" * 80)
        print(open(p, encoding="utf-8").read()[:6000])
    except FileNotFoundError:
        pass